In [1]:
%pip install -q pandas numpy matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\kagoble\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.precision", 4)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

Pull data

In [3]:
# PERERA_URL = (
#     "https://raw.githubusercontent.com/"
#     "Paulilein/Perera2018/main/Perera2018_Data_Original.csv"
# )

# df_raw = pd.read_csv(
#     PERERA_URL,
#     sep=";",
#     decimal=",",
#     encoding="utf-8-sig"
# )

# print("Raw shape:", df_raw.shape)
# df_raw.head()

Raw shape: (5760, 18)


,Reaction_No,Reactant_1_Name,Reactant_1_Short_Hand,Reactant_1_eq,Reactant_1_mmol,Reactant_2_Name,Reactant_2_eq,Catalyst_1_Short_Hand,Catalyst_1_eq,Ligand_Short_Hand,Ligand_eq,Reagent_1_Short_Hand,Reagent_1_eq,Solvent_1_Short_Hand,Product_Yield_PCT_Area_UV,Product_Yield_Mass_Ion_Count,Unnamed: 16,Unnamed: 17
0,1,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,P(tBu)3,0.125,NaOH,2.5,MeCN,4.76,6262.06,NaN,NaN
1,2,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,P(Ph)3,0.125,NaOH,2.5,MeCN,4.12,13245.57,NaN,NaN
2,3,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,AmPhos,0.125,NaOH,2.5,MeCN,2.58,3009.17,NaN,NaN
3,4,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,P(Cy)3,0.125,NaOH,2.5,MeCN,4.44,30860.70,NaN,NaN
4,5,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,P(o-Tol)3,0.125,NaOH,2.5,MeCN,1.95,2486.31,NaN,NaN


In [4]:
# df_raw.to_csv(
#     DATA_DIR / "perera_pfizer_raw_download.csv",
#     index=False
# )

Renaming amd removing space

In [ ]:
# Load the raw data from the local CSV file
df_raw = pd.read_csv(
    DATA_DIR / "perera_pfizer_raw_download.csv"
)

# Remove completely empty "Unnamed" columns
df = df_raw.loc[:, ~df_raw.columns.str.startswith("Unnamed")].copy()

# Remove leading/trailing whitespace from text columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# # Rename columns to notebook-friendly names #consider removing
# rename_map = {
#     "Reaction_No": "reaction_id",
#     "Reactant_1_Name": "reactant1_name",
#     "Reactant_1_Short_Hand": "reactant1_label",
#     "Reactant_1_eq": "reactant1_eq",
#     "Reactant_1_mmol": "reactant1_mmol",
#     "Reactant_2_Name": "reactant2_label",
#     "Reactant_2_eq": "reactant2_eq",
#     "Catalyst_1_Short_Hand": "catalyst",
#     "Catalyst_1_eq": "catalyst_eq",
#     "Ligand_Short_Hand": "ligand",
#     "Ligand_eq": "ligand_eq",
#     "Reagent_1_Short_Hand": "base",
#     "Reagent_1_eq": "base_eq",
#     "Solvent_1_Short_Hand": "solvent",
#     "Product_Yield_PCT_Area_UV": "yield_pct",
#     "Product_Yield_Mass_Ion_Count": "product_ion_count",
# }

# df = df.rename(columns=rename_map)

# Pandas interprets the literal "None" conditions as missing values.
# Restore those as explicit experimental factor levels.
try:
    df["ligand"] = df["ligand"].fillna("None")
except KeyError:
    df["Ligand"] = df["Ligand"].fillna("None")

df["base"] = df["base"].fillna("None")

print(df.shape)
df.head()

SyntaxError: expected 'except' or 'finally' block (281622050.py, line 40)

Removing alternative solvent 

In [15]:
print(df["solvent"].value_counts())

solvent
MeCN    1440
DMF     1440
THF     1344
MeOH    1344
Name: count, dtype: int64


In [14]:
#find rows containing artifacts and drop them
drop_solvent_df = df[df['solvent'].isin(["MeOH/H2O_V2 9:1","THF_V2"])]
df = df.drop(drop_solvent_df.index, axis=0)

In [16]:
print(df["reactant1_name"].value_counts())

reactant1_name
6-chloroquinoline                         1152
6-Bromoquinoline                          1152
6-triflatequinoline                       1152
6-Iodoquinoline                           1152
Potassium quinoline-6-trifluoroborate      384
6-Quinolineboronic acid pinacol ester      384
6-quinoline-boronic acid hydrochloride     192
Name: count, dtype: int64
